In [2]:
import io
import os

import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option('display.width', 200)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
CUTOFF = pd.Timestamp('2026-09-11')          # fecha de referencia del enunciado

In [3]:
#Q1.
url = 'https://www.iposcoop.com/ipos-recently-filed/'
tables = pd.read_html(io.StringIO(requests.get(url, headers=HEADERS).text))
df = tables[0]
df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-28,Encore Medical Inc.,EMI,The Oak Ridge Financial Services Group/Network...,3.0,$5.00,$5.00,$15.00,Wednesday,S/O
1,2026-09-25,Elevation Acquisition Group,ELEVU,Maxim Group,10.0,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.0,$15.00,$18.00,$825.00,Withdrawn,S/O
3,2026-09-25,RegenLab USA,RGNA,StoneX,0.0,NaN,NaN,$35.00,TBA,S/O
4,2026-09-25,TCGX Acquisition Corp. II (Stock-Only),TCXB,Guggenheim Securities,10.0,$10.00,$10.00,$100.00,TBA,S/O


In [4]:
df['File Date'] = pd.to_datetime(df['File Date'], errors='coerce')

withdrawn = df[df['Expected To Trade'] == 'Withdrawn'].copy()
withdrawn = withdrawn[withdrawn['File Date'] < CUTOFF].reset_index(drop=True)

print('Retiradas en la tabla:', (df['Expected To Trade'] == 'Withdrawn').sum())
print('Retiradas antes del corte:', len(withdrawn))

rules = [
    (('Technologies',),                                                    'Technologies'),
    (('Acquisition Corp', 'Acquisition Corporation', 'Corp'),               'Acquisition Corp'),
    (('Inc', 'Incorporated'),                                              'Inc.'),
    (('Group',),                                                           'Group'),
    (('Ltd', 'Limited'),                                                   'Limited'),
    (('Holdings', 'Holding'),                                              'Holdings'),
]

company_types = []
for company in withdrawn['Company']:
    label = 'Other'
    for patterns, tag in rules:
        if any(pattern in company for pattern in patterns):
            label = tag
            break
    company_types.append(label)

withdrawn['Company Type'] = company_types
withdrawn[['File Date', 'Company', 'Company Type']].to_string()

low = pd.to_numeric(withdrawn['Price Low'].astype(str)
                    .str.replace(r'[$,\s]', '', regex=True)
                    .replace('-', np.nan), errors='coerce')
high = pd.to_numeric(withdrawn['Price High'].astype(str)
                     .str.replace(r'[$,\s]', '', regex=True)
                     .replace('-', np.nan), errors='coerce')
est_vol = pd.to_numeric(withdrawn['Est $ Vol (millions)'].astype(str)
                        .str.replace(r'[$,\s]', '', regex=True)
                        .replace('-', np.nan), errors='coerce')
shares = pd.to_numeric(withdrawn['Shares (millions)'], errors='coerce')

n_prices = low.notna().astype(int) + high.notna().astype(int)
withdrawn['Avg_price'] = (low.fillna(0) + high.fillna(0)) / n_prices.replace(0, np.nan)
withdrawn['Shares (millions)'] = shares
withdrawn['Est $ Vol (millions)'] = est_vol

withdrawn[['Company', 'Price Low', 'Price High', 'Avg_price',
           'Shares (millions)', 'Est $ Vol (millions)']].to_string()
by_shares = withdrawn['Shares (millions)'] * withdrawn['Avg_price']
withdrawn['Shares_offered_value'] = by_shares.fillna(withdrawn['Est $ Vol (millions)'])

summary = (withdrawn.groupby('Company Type')['Shares_offered_value']
           .agg(n='count', total_musd='sum')
           .sort_values('total_musd', ascending=False))

summary.round(2)

best_type = summary['total_musd'].idxmax()
print('Tipo con mayor valor total retirado:', best_type)
print('Valor total: %.2f $M' % summary.loc[best_type, 'total_musd'])


Retiradas en la tabla: 35
Retiradas antes del corte: 31
Tipo con mayor valor total retirado: Acquisition Corp
Valor total: 499.99 $M


In [5]:
#Q2
url_2025 = 'https://www.iposcoop.com/2025-pricings/'
tables_2025 = pd.read_html(io.StringIO(requests.get(url_2025, headers=HEADERS).text))
print('Tablas en la página:', len(tables_2025), [list(t.columns) for t in tables_2025])

# Misma precautions: la tabla correcta es la que tiene 'Offer Date'
ipos = next((t for t in tables_2025 if 'Offer Date' in t.columns), None)
if ipos is None:
    raise KeyError('Ninguna tabla tiene "Offer Date". Ejecuta Kernel > Restart and Run All.')

ipos = ipos.copy()
print('Shape:', ipos.shape)
ipos.head()


Tablas en la página: 1 [['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']]
Shape: (231, 10)


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [6]:
ipos['Offer Date'] = pd.to_datetime(ipos['Offer Date'], errors='coerce')
ipos['Return'] = pd.to_numeric(ipos['Return'].astype(str).str.replace('%', '', regex=False),
                               errors='coerce')

ipos_f = ipos[(ipos['Offer Date'] < pd.Timestamp('2025-09-01')) & (ipos['Return'] != 0)].copy()
tickers = ipos_f['Symbol'].tolist()

print('IPO en 2025:', len(ipos))
print('Antes del 01/09/2025 y con retorno != 0:', len(ipos_f))

IPO en 2025: 231
Antes del 01/09/2025 y con retorno != 0: 146


In [7]:
raw = yf.download(tickers, start='2025-01-01', end='2026-09-12',
                  progress=False, auto_adjust=True, threads=True, group_by='ticker')

print('tipo de columnas:', type(raw.columns))
print('niveles:', raw.columns.names)

frames = []
for ticker in tickers:
    if ticker not in raw.columns.get_level_values(0):     # no está en Yahoo -> delisted
        continue
    part = raw[ticker].dropna(how='all')
    if part.empty:
        continue
    part.columns = [c.lower() for c in part.columns]        # open, high, low, close, volume
    part = part.reset_index()
    part.columns = ['date'] + list(part.columns[1:])
    part['ticker'] = ticker
    frames.append(part)

stocks_df = pd.concat(frames, ignore_index=True)

print('Tickers con datos:', stocks_df['ticker'].nunique(), 'de', len(tickers))
print('Filas:', len(stocks_df))
stocks_df.head()

$EMPG: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CAEP"}}}
$PTNM: possibly delisted; no timezone found
$MJID: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$DLXY: possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-12)
$WGRX: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found

15 Failed downloads:
['EMPG', 'AGH', 'PTNM', 'MJID', 'AHL

tipo de columnas: <class 'pandas.MultiIndex'>
niveles: ['Ticker', 'Price']
Tickers con datos: 131 de 146
Filas: 44524


,date,open,high,low,close,volume,ticker
0,2025-08-29,210.5,356.0,209.5,229.5,126794.0,PMI
1,2025-09-02,242.5,267.5,232.5,257.0,98358.0,PMI
2,2025-09-03,257.0,270.0,250.5,264.5,35960.0,PMI
3,2025-09-04,265.5,287.5,265.0,286.5,28074.0,PMI
4,2025-09-05,288.0,295.0,282.5,292.5,18572.0,PMI


In [8]:
stocks_df = stocks_df.sort_values(['ticker', 'date']).reset_index(drop=True)

stocks_df['growth_252d'] = stocks_df.groupby('ticker')['close'].transform(
    lambda s: s / s.shift(252))
stocks_df['volatility'] = stocks_df.groupby('ticker')['close'].transform(
    lambda s: s.rolling(30).std() * np.sqrt(252))
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

stocks_df.tail()

snapshot = stocks_df[stocks_df['date'] == CUTOFF]
stats = snapshot[['growth_252d', 'volatility', 'Sharpe']].describe().T
stats.round(4)

print('Mediana del Sharpe ->', round(snapshot['Sharpe'].median(), 4))

Mediana del Sharpe -> 0.048


C:\Users\victo\AppData\Roaming\Python\Python314\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [9]:
print('Acciones con 252 días de histórico: %d de %d'
      % (snapshot['growth_252d'].notna().sum(), len(snapshot)))
print()
print('growth_252d  ->  mediana %.4f | media %.4f | max %.4f'
      % (snapshot['growth_252d'].median(), snapshot['growth_252d'].mean(),
         snapshot['growth_252d'].max()))
print('Sharpe        ->  mediana %.4f | p25 %.4f | p75 %.4f'
      % (snapshot['Sharpe'].median(), snapshot['Sharpe'].quantile(.25),
         snapshot['Sharpe'].quantile(.75)))

Acciones con 252 días de histórico: 129 de 131

growth_252d  ->  mediana 0.5960 | media 1.0652 | max 33.6383
Sharpe        ->  mediana 0.0480 | p25 0.0119 | p75 0.1342


In [10]:
# La media de Sharpe sale `inf`: tres tickers con volatilidad 0 (precio congelado)
print('Tickers con volatility == 0:', snapshot.loc[snapshot['volatility'] == 0, 'ticker'].tolist())

ranking = snapshot.replace([np.inf, -np.inf], np.nan).dropna(subset=['Sharpe'])
cols = ['ticker', 'growth_252d', 'volatility', 'Sharpe']
print()
print('Mejor Sharpe (retorno bajo, riesgo bajo):')
print(ranking.nlargest(5, 'Sharpe')[cols].round(3).to_string(index=False))
print()
print('Peor Sharpe:')
print(ranking.nsmallest(5, 'Sharpe')[cols].round(3).to_string(index=False))
print()
print('Crecimiento por encima de la media pero Sharpe por debajo de la mediana:')
low = ranking[ranking['growth_252d'] > ranking['growth_252d'].mean()].nsmallest(5, 'Sharpe')
print(low[cols].round(3).to_string(index=False))

Tickers con volatility == 0: ['EFTY', 'MAMK', 'NUTR']

Mejor Sharpe (retorno bajo, riesgo bajo):
ticker  growth_252d  volatility      Sharpe
  MAGH        5.452       0.000 4195710.391
 HCMAU        1.035       0.183       5.385
  CEPF        1.024       0.352       2.771
  TMDE        0.776       0.415       1.751
  TLIH        0.990       0.649       1.449

Peor Sharpe:
ticker  growth_252d  volatility  Sharpe
  LZMH        0.018       0.803  -0.040
  FMFC        0.018       1.001  -0.032
  EPSM        0.034       1.024  -0.016
  TDIC        0.005       3.162  -0.014
  DGNX        0.016       2.409  -0.014

Crecimiento por encima de la media pero Sharpe por debajo de la mediana:
ticker  growth_252d  volatility  Sharpe
  HTFL        1.490     153.351   0.009
  VOYG        1.190      77.146   0.015
  HNGE        1.577      74.141   0.021
   NIQ        1.040      46.475   0.021
  CHYM        1.419      46.857   0.029


In [11]:
#Q3
min_date = stocks_df.groupby('ticker')['date'].min().rename('min_date')
stocks_df = stocks_df.merge(min_date, on='ticker')

entry = stocks_df[stocks_df['date'] == stocks_df['min_date']][['ticker', 'min_date', 'close']]
entry = entry.rename(columns={'close': 'first_close'})

joined = (stocks_df.merge(entry, on=['ticker', 'min_date'], how='inner')
                .sort_values(['ticker', 'date'])
                .reset_index(drop=True))
joined.head()

,date,open,high,low,close,volume,ticker,growth_252d,volatility,Sharpe,min_date,first_close
0,2025-01-24,17.250000,18.000000,16.500000,17.379999,2217300.0,AAPG,NaN,NaN,NaN,2025-01-24,17.379999
1,2025-01-27,17.389999,17.500000,17.000000,17.250000,315800.0,AAPG,NaN,NaN,NaN,2025-01-24,17.379999
2,2025-01-28,17.370001,17.379999,17.025000,17.200001,39800.0,AAPG,NaN,NaN,NaN,2025-01-24,17.379999
3,2025-01-29,17.389999,17.450001,17.190001,17.330000,121000.0,AAPG,NaN,NaN,NaN,2025-01-24,17.379999
4,2025-01-30,17.450001,17.860001,17.280001,17.600000,228900.0,AAPG,NaN,NaN,NaN,2025-01-24,17.379999


In [12]:
for months in range(1, 13):
    forward_close = joined.groupby('ticker')['close'].shift(-21 * months)
    joined['future_growth_%d_m' % months] = forward_close / joined['first_close']

cols = ['future_growth_%d_m' % m for m in range(1, 13)]
growth_stats = joined[cols].describe().T
growth_stats.round(4)

medians = growth_stats['50%']
best_month = int(medians.idxmax().split('_')[2])

print('Mes óptimo:', best_month)
print('Mediana máxima: %.4f  (+%.2f %%)' % (medians.max(), medians.max() * 100))
print()
print('Mediana por horizonte:')
print((medians * 100).round(2).to_string())

for m in [1, 3, 6, 12]:
    col = 'future_growth_%d_m' % m
    sub = joined[joined[col].notna()]
    print('%2d meses -> filas %6d | tickers %3d | mediana %6.3f | media %8.3f'
          % (m, len(sub), sub['ticker'].nunique(), sub[col].median(), sub[col].mean()))

Mes óptimo: 1
Mediana máxima: 0.6320  (+63.20 %)

Mediana por horizonte:
future_growth_1_m     63.20
future_growth_2_m     60.50
future_growth_3_m     58.51
future_growth_4_m     56.33
future_growth_5_m     54.61
future_growth_6_m     52.50
future_growth_7_m     50.01
future_growth_8_m     47.31
future_growth_9_m     44.72
future_growth_10_m    42.18
future_growth_11_m    39.72
future_growth_12_m    38.07
 1 meses -> filas  41773 | tickers 131 | mediana  0.632 | media   28.984
 3 meses -> filas  36306 | tickers 130 | mediana  0.585 | media   22.943
 6 meses -> filas  28116 | tickers 130 | mediana  0.525 | media   17.056
12 meses -> filas  11743 | tickers 129 | mediana  0.381 | media    3.127


In [13]:
#q4
import gdown

file_id = '1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-'
if not os.path.exists('data.parquet'):
    gdown.download('https://drive.google.com/uc?id=%s' % file_id, 'data.parquet', quiet=False)

try:
    panel = pd.read_parquet('data.parquet', engine='pyarrow')
except Exception as err:
    print(type(err).__name__, err)
    print('-> Reinicia el kernel (Kernel > Restart) y vuelve a ejecutar esta celda.')
    raise

print('Filas:', len(panel), '| Columnas:', len(panel.columns))
print([c for c in panel.columns if 'rsi' in c.lower() or 'future' in c.lower()][:10])

panel['Date'] = pd.to_datetime(panel['Date'])

trades = panel[(panel['Date'] >= '2000-01-01')
               & (panel['Date'] < '2025-06-01')
               & (panel['rsi'] < 30)].dropna(subset=['growth_future_30d'])

print('Operaciones:', len(trades))
trades.head()

net_income = 1000 * (trades['growth_future_30d'] - 1).sum()

print('Retorno medio a 30 días: %.4f %%' % ((trades['growth_future_30d'].mean() - 1) * 100))
print('Tasa de acierto:          %.2f %%' % ((trades['growth_future_30d'] > 1).mean() * 100))
print('Capital rotado:           %.0f $' % (1000 * len(trades)))
print('Net income:               %.2f $' % net_income)
print()
print('RESPUESTA (miles de $):   %.2f  ->  %.0f' % (net_income / 1000, net_income / 1000))

Filas: 229932 | Columnas: 203
['growth_future_30d', 'is_positive_growth_30d_future', 'rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']
Operaciones: 5206
Retorno medio a 30 días: 1.2640 %
Tasa de acierto:          55.13 %
Capital rotado:           5206000 $
Net income:               65805.59 $

RESPUESTA (miles de $):   65.81  ->  66
